# Retail & Sales Performance Dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

df = pd.read_csv("blinkit_data.csv")

%matplotlib inline

In [ ]:
df_shape = df.shape
dataType = df.dtypes
print(dataType)
df.head(5)


In [ ]:
# structural errors & Duplicates

df["Item Fat Content"].unique()
for col in df.columns:
    if df[col].nunique() < 20:
        print(df[col].value_counts())
        print("---"*50)


df["Item Fat Content"] = df["Item Fat Content"].replace({'low fat': "Low Fat", "LF": "Low Fat", "reg": "Regular"})
df.head(10)

# df.duplicated().sum()
# df[df.duplicated()]


In [ ]:
# df[df.isnull().any(axis=1)] # Returns rows with Null values in any column

df.groupby("Item Type").agg(missing_item_weight = ("Item Weight", lambda x: x.isnull().sum()),
                            median_of_each_type = ("Item Weight", "median"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Histogram (Distribution dekhne ke liye)
sns.histplot(df["Item Weight"], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Item Weight Distribution")

# 2. Boxplot (Outliers dekhne ke liye)
sns.boxplot(x=df["Item Weight"], ax=axes[1], color='lightcoral')
axes[1].set_title("Item Weight Boxplot (Outliers)")

plt.tight_layout()
plt.show()

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
# X-axis par Item Type aur Y-axis par Item Weight
sns.boxplot(data=df, x="Item Type", y="Item Weight")
plt.xticks(rotation=90) # Taaki naam aapas mein takraye nhi

In [ ]:
## filing the missing values in item weight feature 

df["Item Weight"] = df["Item Weight"].fillna(df.groupby("Item Type")["Item Weight"].transform("median"))

#df.isnull().sum()

## Understanding the dataset
This Retail and Sales performance dataset can be divided into 3 parts:

1. Product( Item ) Information: which includes columns like( Item Identifier(Unique ID), Item Type (Categories), Item Fat Content, Item Weight, and Item Visibility).

2. Store (Outlet) Information: Outlet identifier, Outlet Establishment Year, Outlet Size, Outlet Location Type, Outlet Type

3. Performance Metrics (Target Variables): Sales and Rating

## Missing Values Analysis

In [ ]:
# Total missing

print(f"Total no. of missing values in DataFrame: \n{df.isnull().sum()}\n\n")

print(f"Total percentage of missing values: \n{df.isnull().mean()  * 100}")

In [ ]:
# Rows with any null
print(df.isnull().any(axis=1).sum())

# # Rows with ALL null
# print(df.isnull().all(axis=1).sum())

# Columns with missing > threshold
#print(df.columns[df.isnull().mean() > 0.3]) # >30% missing 


# Missing Bar 
missing = df.isnull().sum().sort_values(ascending=False)
# missing = missing[missing > 0]
# fig, ax = plt.subplots(figsize=(10,5))

# bars = ax.bar(missing.index, missing.values, color='tomato', edgecolor='white')
# plt.xticks(rotation=45, ha='right')
# ax.set_ylabel("Count")
# for bar in bars:
#      ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(bar.get_height()), ha='center', fontsize=9)

# plt.tight_layout()
# plt.show()

# Missing pattern heatmap


## Filling Missing Values 

In [ ]:
# Har 'Item Type' ka average weight nikal kar missing values ko fill karna
df['Item Weight'] = df.groupby('Item Type')['Item Weight'].transform(lambda x: x.fillna(x.mean()))

# Checking if any row is null or not.
#print(df['Item Weight'].isnull().sum())

In [ ]:
# 1. Missing Values
# df.duplicated().sum() # check any duplicate row
# df = df.drop_duplicates() # to delete duplicates

# 2. Inconsistent Data / Structural Errors
distinct_values = df["Item Fat Content"].value_counts() # To Check
df["Item Fat Content"] = df["Item Fat Content"].replace({"LF":"Low Fat", "reg":"Regular", "low fat":"Low Fat"}) # To Fix
print(distinct_values)


In [ ]:
# 3. Incorrect Data Types
#print(df.dtypes)

df["Outlet Establishment Year"] = pd.to_datetime(df["Outlet Establishment Year"],format="%Y" , errors="coerce").dt.to_period("Y")    # converting dtype to datetime64[s]

# 4. Outliers
plt.boxplot(df["Sales"])
plt.title('Sales Column mein Outliers Check Karna')
plt.show()

In [ ]:
df.head(10)

## Business problems & requirements
1. *Sales Optimization:* Kaun se stores ya products sabse zyada revenue generate kar rahe hain aur kyun?
2. *Inventory Management:* Kis item type ka weight ya fat content zyada bikta hai, taaki stock maintain rahe?
3. *Visibility vs Sales:* Kya app par kisi item ko zyada dikhane (Item Visibility) se uski sales sach mein badhti hai?
4. *Store Expansion Strategy:* Tier 1 cities ke medium stores zyada paise kama rahe hain ya Tier 3 ke grocery stores?

In [ ]:

# ==========================================
# 📦 INVENTORY MANAGEMENT ANALYSIS
# ==========================================

# 1. Fat Content ke hisab se Total aur Average Sales
fat_content_analysis = df.groupby("Item Fat Content")["Sales"].agg(['sum', 'mean']).sort_values(by='sum', ascending=False)

# 2. Item Type ke sath Fat Content ka combination (Deep Dive)
item_fat_inventory = df.groupby(["Item Type", "Item Fat Content"])["Sales"].sum().unstack().fillna(0)

# 3. Overall Average Weight
avg_weight = df["Item Weight"].mean()

print("🥛 FAT CONTENT VS SALES PERFORMANCE:")
print("-" * 50)
print(fat_content_analysis.round(2).to_string())

print("\n🍎 ITEM TYPE + FAT CONTENT INVENTORY VALUE (Total Sales):")
print("-" * 60)
print(item_fat_inventory.round(2).to_string())

print(f"\n📦 Standard Stock Weight (Average Item Weight): {avg_weight:.2f} kg")


# 1. Weights ko 3 brackets mein baantein (Low, Medium, High Weight)
# Maan lijiye min weight 4kg hai aur max 22kg
df['Weight_Category'] = pd.cut(df['Item Weight'], 
                               bins=[0, 10, 18, 100], 
                               labels=['Light (0-10kg)', 'Medium (10-18kg)', 'Heavy (18kg+)'])

# 2. Check karein ki kis weight category ki sales sabse zyada hai
weight_sales = df.groupby('Weight_Category', observed=False)['Sales'].sum().sort_values(ascending=False)

# 3. Item Type ke sath jod kar dekhein (Deep Dive)
item_weight_matrix = df.groupby(['Item Type', 'Weight_Category'], observed=False)['Sales'].mean().unstack().fillna(0)

print("⚖️ WEIGHT CATEGORY VS TOTAL SALES:")
print("-" * 50)
print(weight_sales.round(2).to_string())

print("\n🍎 KIS ITEM TYPE MEIN KAUNSA WEIGHT ZYADA BIKTA HAI (Avg Sales):")
print("-" * 70)
print(item_weight_matrix.round(2).to_string())

In [ ]:
# Store Expansion Strategy
# 1. Pehle data ko filter karo taaki sirf Tier 1 aur Tier 3 ka data bache
filtered_df = df[df["Outlet Location Type"].isin(["Tier 1", "Tier 3"])]

# 2. Ab teenon columns ke hisab se group karke Average Sales nikaalo
expansion_strategy = filtered_df.groupby(["Outlet Location Type", "Outlet Size", "Outlet Type"])["Sales"].mean().sort_values(ascending=False)

print("🏢 STORE EXPANSION STRATEGY (Filtered for Tier 1 & Tier 3):")
print("-" * 75)
print(expansion_strategy.round(2).to_string())

### Key Performance Indicators (KPIs)

Aapko is data se ye main cheezein calculate karni hain:

* *Total & Average Sales:* Alag-alag outlets aur item types ki total kamai.
* *Average Rating:* Customers kis outlet ya product category se sabse zyada khush hain.
* *Product Visibility Index:* Kya high visibility ka sales ke sath koi positive connection (correlation) hai?
* *Store Age vs Sales:* Purane stores vs Naye stores ki sales ka muqabla.


In [ ]:
# # Total & average Sales

# total_sales = df.Sales.sum()
# sales_by_outlets = df.groupby("Outlet Type").Sales.sum()
# sales_by_itemType = df.groupby("Item Type").Sales.sum()
# print(f'Total Sales: \n{total_sales}\n{"-"*50}\nTotal Sales by Outlets Type: \n{sales_by_outlets}\n{"-"*50}\nTotal Sales by Item Types: \n{sales_by_itemType}')


# # Total & Average Sales (Slightly Upgraded)

total_sales = df["Sales"].sum()
print(f"💰 Global Total Sales: {total_sales:,.2f}\n" + "="*50)

# .agg(['sum', 'mean']) karne se ek hi baar mein dono tables ban jayengi
sales_by_outlets = df.groupby("Outlet Type")["Sales"].agg(['sum', 'mean']).sort_values(by='sum', ascending=False)
sales_by_itemType = df.groupby("Item Type")["Sales"].agg(['sum', 'mean']).sort_values(by='sum', ascending=False)

print("🏪 Sales by Outlet Type (Total & Mean):")
print(sales_by_outlets.round(2).to_string())
print("-" * 50)

print("\n🍎 Sales by Item Types (Total & Mean):")
print(sales_by_itemType.round(2).to_string())

In [ ]:
# Average Rating
average_rating = df.Rating.mean()
rating_by_outletType = df.groupby("Outlet Type").Rating.mean().sort_values(ascending=False)
rating_by_itemType = df.groupby("Item Type").Rating.mean().sort_values(ascending=False)
# print(f'Average Rating: {average_rating.round(2)}\n{"-"*50}\n Rating by Outlet Type : {rating_by_outletType.round(2).to_string()}\n{"-"*50}\nRating by Item Type: {rating_by_itemType.round(2).to_string()}')

print("🏪 Average se behtar Outlets:")
print("-" * 30)
# .items() lagane se 'label' mein naam aayega aur 'rating' mein number
for label, rating in rating_by_outletType.items():
    if rating > average_rating:
        print(f"{label}: {rating:.2f} ⭐")

print("\n🍎 Average se behtar Items:")
print("-" * 30)
for label, rating in rating_by_itemType.items():
    if rating > average_rating:
        print(f"{label}: {rating:.2f} ⭐")

In [ ]:
# Product visibility index
visibility_sales_corr = df["Item Visibility"].corr(df["Sales"])

print(f"Product Visibility Index (Correlation): {visibility_sales_corr:.4f}")

# CONCLUSION { (Correlation): -0.0013 }
#  '''High visibility ka sales ke sath KOI positive connection nahi hai. 
#     In fact, dono ke beech mein Zero Correlation (ya bohot hi negligible negative connection) hai. 
#     Iska business matlab yeh hai ki aap product ka visibility score kitna bhi badha dein ya kam kar dein, 
#     uska Sales par koi asar nahi pad raha hai. Dono aapas mein independent hain.'''

In [ ]:
# *Store Age vs Sales:* Purane stores vs Naye stores ki sales ka muqabla.


# Har saal khule stores ki average sales nikalna aur sort karna
store_age_comparison = df.groupby("Outlet Establishment Year")["Sales"].mean().sort_values(ascending=False)

print("🏪 STORE ESTABLISHMENT YEAR VS AVERAGE SALES:")
print("-" * 50)
print(store_age_comparison.round(2).to_string())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Background grid set karna (Sirf Y-axis par lines dikhenge)
sns.set_theme(style="whitegrid") 

sns.barplot(
    data=df, 
    x="Outlet Establishment Year", 
    y="Sales", 
    estimator='mean', 
    errorbar=None, 
    palette="viridis", 
    order=sorted(df["Outlet Establishment Year"].unique())
)

plt.title("Store Establishment Year vs Average Sales 🏪💸", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Establishment Year (Purane -> Naye)", fontsize=12)
plt.ylabel("Average Sales", fontsize=12)

# Faltu ke top aur right borders hatane ke liye
sns.despine(left=True, bottom=True) 

plt.tight_layout() # Labels katne se bachane ke liye
plt.show()